# Non-Linear Least Squares, Render-and-Compare, and Diffusion into Noise

Prepared for Fall 2026. Reference: the notes for Lecture 18 (Fitting SMPL to Images: Optimization and Transformers) and Lecture 19 (Approximate Analysis by Synthesis); the last section (Question 11.3, diffusing an image into noise) uses the notes for Lecture 15 (Computer Vision and Cognitive Science, Part 2). If you find a bug in this notebook, please email the TAs or tell us at office hours.

**Before you start.** Everything is simulated with numpy under fixed seeds (no data files), so you get our numbers. The three questions ask for six figures (one of them with two panels) and about thirty numbers; each bullet says which, and every one of them must appear in your PDF: to vary a parameter, copy the cell below the question (or write a loop) rather than re-running it above.

## Utilities

In [ ]:
from __future__ import annotations
import math

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

%matplotlib inline
np.set_printoptions(precision=4, suppress=True)

COLOUR = {"Gauss-Newton": "tab:orange", "Levenberg-Marquardt": "tab:blue", "squared cost": "tab:blue", "gradient descent": "olive",
          "truth": "k", "Geman-McClure": "tab:purple", "loss": "tab:blue", "histogram": "0.45"}
Z_COLOURS = ["tab:red", "tab:green", "gold"]          # Z = 0 predicted occluded, 1 visible object, 2 background (the notes' code)


def fmt(values, sig: int = 4) -> str:
    """Format a sequence of numbers with `sig` significant digits, separated by spaces."""
    return " ".join(f"{v:.{sig}g}" for v in np.atleast_1d(values))


# ------------------------------------------------------------------ 1. The toy arm: non-linear least squares

LENGTHS = np.array([1.0, 0.8, 0.6])          # the three links
X_TRUE = np.array([0.5, 0.9, -0.8])          # the joint angles (rad) that produced the observations
X_START = np.array([-2.0, 1.5, 2.0])         # the start of every method


def arm_points(x, lengths=LENGTHS):
    """The ends of the links for the joint angles x (each relative to the previous link), shoulder at the origin.
    Returns an (n_links, 2) array: row k is the end of link k."""
    angles = np.cumsum(np.asarray(x, dtype=float))                 # absolute angle of each link
    steps = np.stack([lengths * np.cos(angles), lengths * np.sin(angles)], axis=1)
    return np.cumsum(steps, axis=0)


def residual(x, obs):
    """The residual vector e = predicted - observed link ends, stacked (x_1, y_1, x_2, y_2, x_3, y_3)."""
    return (arm_points(x) - np.asarray(obs, dtype=float)).ravel()


def error(x, obs) -> float:
    """e(x) = e^T e, the sum of the squared distances between predicted and observed link ends."""
    e = residual(x, obs)
    return float(e @ e)


def jacobian(x, lengths=LENGTHS):
    """The (2N x d) Jacobian of the residual vector at x, analytic: parameter j turns every link from j on, so
    d(end of link k)/dx_j = sum_{i=j}^{k} l_i (-sin a_i, cos a_i) for j <= k and 0 for j > k (a_i the absolute angles)."""
    x = np.asarray(x, dtype=float)
    angles = np.cumsum(x)
    d = len(x)
    J = np.zeros((2 * d, d))
    for k in range(d):
        for j in range(k + 1):
            for i in range(j, k + 1):
                J[2 * k, j] += -lengths[i] * np.sin(angles[i])
                J[2 * k + 1, j] += lengths[i] * np.cos(angles[i])
    return J


def finite_difference_jacobian(x, h: float = 1e-6):
    """Central finite differences of arm_points(x).ravel(), used only to check `jacobian`."""
    x = np.asarray(x, dtype=float)
    J = np.zeros((2 * len(x), len(x)))
    for c in range(len(x)):
        step = np.zeros(len(x))
        step[c] = h
        J[:, c] = (arm_points(x + step).ravel() - arm_points(x - step).ravel()) / (2 * h)
    return J


def gauss_newton(obs, x0, n_iter: int = 12):
    """Gauss-Newton: solve J^T J dx = -J^T e (by least squares, so a singular J^T J does not crash) and x <- x + dx.
    Returns the errors e(x) at iterations 0..n_iter (array of n_iter + 1) and the path of parameters (n_iter + 1, d)."""
    x = np.array(x0, dtype=float)
    errors, path = [error(x, obs)], [x.copy()]
    for _ in range(n_iter):
        e, J = residual(x, obs), jacobian(x)
        dx = np.linalg.lstsq(J.T @ J, -J.T @ e, rcond=None)[0]
        x = x + dx
        errors.append(error(x, obs))
        path.append(x.copy())
    return np.array(errors), np.array(path)


def levenberg_marquardt(obs, x0, n_iter: int = 12, mu0: float = 1.0, factor: float = 10.0, mu_max: float = 1e8):
    """Levenberg-Marquardt: dx = -(J^T J + mu I)^{-1} J^T e. A step is accepted only if it reduces e; then mu is divided by
    `factor`. Otherwise mu is multiplied by `factor` and the step tried again (the retries do not count as iterations).
    If mu would exceed mu_max no step is acceptable: the method stops and the remaining iterations keep x.
    Returns the errors (n_iter + 1), the path (n_iter + 1, d), the mu of every accepted step and the number of retries before it."""
    x = np.array(x0, dtype=float)
    mu = mu0
    errors, path, mus, retries = [error(x, obs)], [x.copy()], [], []
    for it in range(n_iter):
        e, J = residual(x, obs), jacobian(x)
        n_retry = 0
        while True:
            dx = np.linalg.lstsq(J.T @ J + mu * np.eye(len(x)), -J.T @ e, rcond=None)[0]
            x_new = x + dx
            if error(x_new, obs) < errors[-1]:                 # accepted: e decreases
                x = x_new
                mus.append(mu)
                retries.append(n_retry)
                mu = mu / factor
                break
            mu = mu * factor                                   # rejected: raise mu and try again
            n_retry += 1
            if mu > mu_max:
                break
        if mu > mu_max:
            print(f"Levenberg-Marquardt: no acceptable step at iteration {it + 1} (mu would exceed {mu_max:g}); "
                  f"e cannot be reduced further, the remaining iterations keep x")
            errors.extend([errors[-1]] * (n_iter - it))
            path.extend([x.copy()] * (n_iter - it))
            break
        errors.append(error(x, obs))
        path.append(x.copy())
    return np.array(errors), np.array(path), np.array(mus), np.array(retries)


def gradient_descent(obs, x0, n_iter: int = 12, step: float = 0.15):
    """Gradient descent with a fixed step on J^T e, the gradient of e/2: x <- x - step * J^T e.
    Returns the errors (n_iter + 1) and the path (n_iter + 1, d)."""
    x = np.array(x0, dtype=float)
    errors, path = [error(x, obs)], [x.copy()]
    for _ in range(n_iter):
        e, J = residual(x, obs), jacobian(x)
        x = x - step * (J.T @ e)
        errors.append(error(x, obs))
        path.append(x.copy())
    return np.array(errors), np.array(path)


def geman_mcclure(r, sigma: float):
    """rho(r) = sigma^2 r^2 / (sigma^2 + r^2) per coordinate: like r^2 for small r, levelling off at sigma^2."""
    r = np.asarray(r, dtype=float)
    return sigma**2 * r**2 / (sigma**2 + r**2)


def geman_mcclure_derivative(r, sigma: float):
    """rho'(r) = 2 sigma^4 r / (sigma^2 + r^2)^2."""
    r = np.asarray(r, dtype=float)
    return 2 * sigma**4 * r / (sigma**2 + r**2) ** 2


def robust_fit(obs, x0, sigma: float = 0.3, n_iter: int = 1000, step: float = 0.1):
    """Gradient descent on the robust cost sum_i rho(e_i) over the six residual coordinates, gradient J^T rho'(e), from x0.
    Start from the squared-cost fit so that the descent lands in the minimum where the third link stays with the inliers;
    from x_0 = (-2, 1.5, 2) it converges to another local minimum (cost 0.0887) in which the third link swings toward the
    outlier. The step 0.1 is below the stability limit of this cost (rho''(0) = 2, twice as stiff as e/2); a step of 0.15
    ends in a two-point oscillation instead of converging.
    Returns the final x and the costs at iterations 0..n_iter."""
    x = np.array(x0, dtype=float)
    costs = [float(np.sum(geman_mcclure(residual(x, obs), sigma)))]
    for _ in range(n_iter):
        e, J = residual(x, obs), jacobian(x)
        x = x - step * (J.T @ geman_mcclure_derivative(e, sigma))
        costs.append(float(np.sum(geman_mcclure(residual(x, obs), sigma))))
    return x, np.array(costs)


def joint_errors(x, x_true=X_TRUE):
    """Distances between the fitted and the true link ends: (elbow, wrist, end of the third link)."""
    return np.linalg.norm(arm_points(x) - arm_points(x_true), axis=1)


def plot_errors(curves: dict, title: str = "e(x) at each iteration, from the same start"):
    """e against the iteration on a logarithmic axis, one curve per method (colours as in the notes)."""
    fig, ax = plt.subplots(figsize=(6.5, 3.4))
    for name, errors in curves.items():
        ax.semilogy(np.arange(len(errors)), errors, "o-", ms=4, color=COLOUR.get(name, None), label=name)
    ax.set_xlabel("iteration")
    ax.set_ylabel("e(x)")
    ax.set_title(title)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


def draw_arm(ax, x, color, lw: float = 2.0, alpha: float = 1.0, ls: str = "-", label=None):
    """One arm: the shoulder (square) and the three links with dots at the joints."""
    pts = np.vstack([np.zeros((1, 2)), arm_points(x)])
    ax.plot(pts[:, 0], pts[:, 1], ls, color=color, lw=lw, alpha=alpha, marker="o", ms=4, label=label)
    ax.plot([0], [0], "s", color=color, ms=7, alpha=alpha)


def plot_arm_fit(path, obs, title: str = "the arm during the Levenberg-Marquardt fit"):
    """The arm at every parameter vector of the path, from the lightest (start) to the darkest (converged); observed link ends as x."""
    fig, ax = plt.subplots(figsize=(4.8, 4.8))
    shades = plt.cm.Blues(np.linspace(0.3, 1.0, len(path)))
    for x, shade in zip(path, shades):
        draw_arm(ax, x, shade, lw=1.5)
    obs = np.asarray(obs)
    ax.plot(obs[:, 0], obs[:, 1], "x", color="k", ms=10, mew=2, label="observed link ends")
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title + "\n(lightest = start, darkest = converged)")
    ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    plt.show()


def plot_arms(fits: dict, obs, x_true=X_TRUE, title: str = "fitted arms over the true one"):
    """The true arm (black, dashed), the fitted arms of the dict name -> x, and the observed link ends (x)."""
    fig, ax = plt.subplots(figsize=(4.8, 4.8))
    draw_arm(ax, x_true, COLOUR["truth"], lw=1.5, ls="--", label="true arm")
    for name, x in fits.items():
        draw_arm(ax, x, COLOUR.get(name, None), lw=2.0, label=name)
    obs = np.asarray(obs)
    ax.plot(obs[:, 0], obs[:, 1], "x", color="k", ms=10, mew=2, label="observed link ends")
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


# ------------------------------------------------------------------ 2. Toy neural mesh model and render-and-compare

GRID = 24
CENTRE = (GRID - 1) / 2.0
_rows, _cols = np.meshgrid(np.arange(GRID), np.arange(GRID), indexing="ij")
CELL_XY = np.stack([_cols - CENTRE, _rows - CENTRE], axis=-1).reshape(-1, 2)     # cell centres: x = column, y = row (origin lower)


def make_mesh(K: int = 12, D: int = 8, r_min: float = 5.0, r_max: float = 8.0, feature_norm: float = 2.0, seed: int = 0) -> dict:
    """K vertices on an irregular polygon around the grid's centre (angles 360/K degrees apart, each moved by up to 10 degrees;
    radii uniform in [r_min, r_max] cells), each with a fixed random feature theta_r in R^D of length feature_norm; the clutter
    feature beta = 0. Returns {"vertices": (K, 2) relative to the centre, "theta": (K, D), "beta": (D,)}."""
    rng = np.random.default_rng(seed)
    angles = np.deg2rad(np.arange(K) * 360.0 / K + rng.uniform(-10.0, 10.0, size=K))
    radii = rng.uniform(r_min, r_max, size=K)
    vertices = np.stack([radii * np.cos(angles), radii * np.sin(angles)], axis=1)
    theta = rng.normal(size=(K, D))
    theta = feature_norm * theta / np.linalg.norm(theta, axis=1, keepdims=True)
    return {"vertices": vertices, "theta": theta, "beta": np.zeros(D)}


def rotate(points, phi: float):
    """Rotate (n, 2) points about the origin by phi degrees (counter-clockwise)."""
    a = np.deg2rad(phi)
    R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
    return points @ R.T


def inside_polygon(points, polygon):
    """Which of the (P, 2) points lie inside the (K, 2) polygon (even-odd rule, vectorised over points and edges)."""
    x, y = points[:, 0:1], points[:, 1:2]
    xa, ya = polygon[:, 0][None, :], polygon[:, 1][None, :]
    xb, yb = np.roll(polygon[:, 0], -1)[None, :], np.roll(polygon[:, 1], -1)[None, :]
    crosses = (ya > y) != (yb > y)                                        # the edge spans the point's y
    denom = np.where(yb != ya, yb - ya, 1.0)                              # horizontal edges never cross: avoid 0/0
    x_at_y = xa + (y - ya) * (xb - xa) / denom
    return (np.sum(crosses & (x < x_at_y), axis=1) % 2) == 1


def render(mesh: dict, phi: float):
    """Render the mesh at the pose phi (degrees): every cell inside the rotated polygon carries the feature of its nearest
    vertex, every other cell the clutter feature beta. Returns the feature map (GRID, GRID, D), the foreground mask
    (GRID, GRID) and the index of the vertex rendered at each cell (-1 outside)."""
    verts = rotate(mesh["vertices"], phi)
    inside = inside_polygon(CELL_XY, verts).reshape(GRID, GRID)
    nearest = np.argmin(np.sum((CELL_XY[:, None, :] - verts[None, :, :]) ** 2, axis=-1), axis=1).reshape(GRID, GRID)
    F = np.where(inside[..., None], mesh["theta"][nearest], mesh["beta"])
    return F, inside, np.where(inside, nearest, -1)


def nll(F, mesh: dict, phi: float) -> float:
    """L_ML(phi) = 1/2 sum_FG |f_i - theta_r|^2 + 1/2 sum_BG |f_i' - beta|^2 (all spreads 1, constants dropped)."""
    predicted, inside, _ = render(mesh, phi)
    return 0.5 * float(np.sum((F - predicted) ** 2))


def nll_with_z(F, mesh: dict, phi: float, pi: float = 0.6):
    """The loss with the occlusion variables: at every foreground cell the foreground score -1/2 |f_i - theta_r|^2 + log pi
    competes with the background score -1/2 |f_i - beta|^2 + log(1 - pi); the larger wins and sets z_i (1 if the
    foreground wins). Background cells cost 1/2 |f_i' - beta|^2 as before. Returns the loss (minus the total score) and the
    map Z with 0 = predicted occluded (z_i = 0), 1 = visible object (z_i = 1), 2 = background."""
    if not 0.0 < pi < 1.0:
        raise ValueError(f"the prior pi = p(z_i = 1) must lie strictly between 0 and 1; got {pi}")
    predicted, inside, _ = render(mesh, phi)
    fg_score = -0.5 * np.sum((F - predicted) ** 2, axis=-1) + math.log(pi)
    bg_cost = 0.5 * np.sum((F - mesh["beta"]) ** 2, axis=-1)
    bg_score = -bg_cost + math.log(1.0 - pi)
    visible = fg_score >= bg_score
    Z = np.full((GRID, GRID), 2, dtype=int)
    Z[inside & visible] = 1
    Z[inside & ~visible] = 0
    loss = -float(np.sum(np.maximum(fg_score, bg_score)[inside])) + float(np.sum(bg_cost[~inside]))
    return loss, Z


def wrap(phi):
    """An angle in degrees brought into [-180, 180)."""
    return ((np.asarray(phi, dtype=float) + 180.0) % 360.0) - 180.0


def search_pose(F, mesh: dict, phi0: float, loss=nll, step: float = 4.0, min_step: float = 0.25):
    """Search on the angle: compare the loss at phi - step and phi + step with the loss at phi; move to the better neighbour
    if it improves, otherwise halve the step; stop when the step falls below min_step (> 0). Angles are taken modulo 360.
    `loss` is any function (F, mesh, phi) -> number. Returns the angle, its loss and the number of loss evaluations."""
    if not min_step > 0:
        raise ValueError(f"min_step must be positive (the step is halved until it falls below min_step); got {min_step}")
    if not step > 0:
        raise ValueError(f"step must be positive; got {step}")
    phi = float(wrap(phi0))
    current = loss(F, mesh, phi)
    n_eval = 1
    while step >= min_step:
        candidates = [float(wrap(phi - step)), float(wrap(phi + step))]
        values = [loss(F, mesh, c) for c in candidates]
        n_eval += 2
        k = int(np.argmin(values))
        if values[k] < current:
            phi, current = candidates[k], values[k]
        else:
            step /= 2.0
    return phi, current, n_eval


def best_of_starts(F, mesh: dict, n_starts: int = 12, loss=nll):
    """Evaluate the loss at n_starts angles spaced uniformly over the circle and search from the best of them."""
    starts = wrap(np.arange(n_starts) * 360.0 / n_starts)
    values = [loss(F, mesh, float(s)) for s in starts]
    return search_pose(F, mesh, float(starts[int(np.argmin(values))]), loss)


def pose_error(phi: float, phi_true: float) -> float:
    """The angle between the estimated and the true rotation, in degrees in [0, 180]."""
    return float(abs(wrap(phi - phi_true)))


def accuracy_measures(errors) -> dict:
    """ACC_pi/6 and ACC_pi/18 (percentages of pose errors below 30 and 10 degrees) and MedErr (median error, degrees)."""
    errors = np.asarray(errors, dtype=float)
    if errors.size == 0:
        raise ValueError("accuracy_measures needs at least one pose error")
    return {"ACC_pi/6": 100.0 * float(np.mean(errors < 30.0)), "ACC_pi/18": 100.0 * float(np.mean(errors < 10.0)),
            "MedErr": float(np.median(errors))}


def make_target(mesh: dict, phi_true: float, noise: float = 0.3, seed: int = 1):
    """A target feature map: the rendering at phi_true plus Gaussian noise of s.d. `noise` on every coordinate of every cell.
    Returns the map and the true foreground mask."""
    rng = np.random.default_rng(seed)
    F, inside, _ = render(mesh, phi_true)
    return F + rng.normal(0.0, noise, size=F.shape), inside


def add_occluder(F, inside, level: float = 0.3, noise: float = 0.3, seed: int = 2):
    """Paste a square of random unit features (plus the same noise) over the target, centred on a random object cell and
    grown until it covers at least `level` of the object's cells. Returns the occluded map, the mask of the occluded object
    cells, the mask of the whole square and the fraction of the object covered."""
    if not 0.0 < level <= 1.0:
        raise ValueError(f"the occlusion level must lie in (0, 1]; got {level}")
    rng = np.random.default_rng(seed)
    object_cells = np.argwhere(inside)
    centre = object_cells[rng.integers(len(object_cells))]
    half = 0
    while True:
        square = np.zeros((GRID, GRID), dtype=bool)
        square[max(centre[0] - half, 0):centre[0] + half + 1, max(centre[1] - half, 0):centre[1] + half + 1] = True
        occluded = square & inside
        if occluded.sum() >= level * len(object_cells) or half >= GRID:
            break
        half += 1
    u = rng.normal(size=(int(square.sum()), F.shape[-1]))
    u = u / np.linalg.norm(u, axis=1, keepdims=True)
    F_occ = F.copy()
    F_occ[square] = u + rng.normal(0.0, noise, size=u.shape)
    return F_occ, occluded, square, occluded.sum() / len(object_cells)


def plot_loss_curve(angles, losses, phi_true: float, title: str = r"$L_{ML}$ against the pose angle", ylabel: str = r"$L_{ML}(\varphi)$"):
    """The loss against the angle (blue, as in the notes), the true angle (black dashed) and the minimum (dot)."""
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    ax.plot(angles, losses, color=COLOUR["loss"], lw=1.5, label="loss")
    ax.axvline(phi_true, color="k", ls="--", lw=1, label=f"true angle {phi_true:g}°")
    k = int(np.argmin(losses))
    ax.plot([angles[k]], [losses[k]], "o", color="k", ms=5, label=f"minimum at {angles[k]:g}°")
    ax.set_xlabel(r"pose angle $\varphi$ (degrees)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


def show_feature_norm(F, title: str, vmax: float = 3.0, mesh: dict = None, phi: float = None):
    """The length |f_i| of the feature at every cell (vmin 0, vmax fixed), with the polygon drawn if mesh and phi are given."""
    fig, ax = plt.subplots(figsize=(4.2, 3.8))
    im = ax.imshow(np.linalg.norm(F, axis=-1), origin="lower", cmap="viridis", vmin=0.0, vmax=vmax,
                   extent=(-CENTRE - 0.5, CENTRE + 0.5, -CENTRE - 0.5, CENTRE + 0.5))
    if mesh is not None and phi is not None:
        v = rotate(mesh["vertices"], phi)
        v = np.vstack([v, v[:1]])
        ax.plot(v[:, 0], v[:, 1], "w-", lw=1, marker="o", ms=3)
    ax.set_xlabel("x (cells)")
    ax.set_ylabel("y (cells)")
    ax.set_title(title, fontsize=10)
    fig.colorbar(im, ax=ax, label=r"$|f_i|$")
    fig.tight_layout()
    plt.show()


def plot_z(maps: dict):
    """Occlusion maps side by side (dict title -> Z) in the colour code of the notes: red predicted occluded, green visible
    object, yellow background."""
    cmap = ListedColormap(Z_COLOURS)
    fig, axes = plt.subplots(1, len(maps), figsize=(3.6 * len(maps) + 1.5, 3.6))
    for ax, (title, Z) in zip(np.atleast_1d(axes), maps.items()):
        ax.imshow(Z, origin="lower", cmap=cmap, vmin=-0.5, vmax=2.5, interpolation="nearest",
                  extent=(-CENTRE - 0.5, CENTRE + 0.5, -CENTRE - 0.5, CENTRE + 0.5))
        ax.set_title(title)
        ax.set_xlabel("x (cells)")
        ax.set_ylabel("y (cells)")
    handles = [Patch(color=Z_COLOURS[0], label="predicted occluded ($z_i = 0$)"), Patch(color=Z_COLOURS[1], label="visible object ($z_i = 1$)"),
               Patch(color=Z_COLOURS[2], label="background")]
    fig.legend(handles=handles, fontsize=8, loc="center right")
    fig.tight_layout(rect=(0, 0, 0.78, 1))
    plt.show()


# ------------------------------------------------------------------ 3. Diffusing into noise and back

def linear_schedule(T: int = 1000, beta_1: float = 1e-4, beta_T: float = 0.02):
    """The noise variances beta_s rising linearly from beta_1 to beta_T over T steps and gamma_t = prod_{s<=t} (1 - beta_s).
    Returns betas (T,) and gammas (T,); index t - 1 holds step t."""
    betas = np.linspace(beta_1, beta_T, T)
    return betas, np.cumprod(1.0 - betas)


def gamma_at(t: int, gammas) -> float:
    """gamma_t for t = 0 (no noise: 1), ..., T."""
    if t < 0 or t > len(gammas):
        raise ValueError(f"t must lie in 0..{len(gammas)}; got {t}")
    return 1.0 if t == 0 else float(gammas[t - 1])


def diffuse(I0, t: int, gammas, rng):
    """I_t = sqrt(gamma_t) I_0 + sqrt(1 - gamma_t) eps with a fresh draw of eps ~ N(0, 1) per sample."""
    g = gamma_at(t, gammas)
    return np.sqrt(g) * np.asarray(I0, dtype=float) + np.sqrt(1.0 - g) * rng.normal(size=np.shape(I0))


def oracle_I0(It, gamma: float):
    """The exact expected value of I_0 given I_t for I_0 = +-1 with probability 1/2 each: tanh(sqrt(gamma) I_t / (1 - gamma))."""
    return np.tanh(np.sqrt(gamma) * np.asarray(It, dtype=float) / max(1.0 - gamma, 1e-12))


def reconstruct(I_T, gammas, n_steps: int = 30):
    """From I_T (pure noise) down to t = 0 in n_steps equal steps of t: at each level estimate I_0 by `oracle_I0`, solve
    I_t = sqrt(gamma_t) I_0 + sqrt(1 - gamma_t) eps for eps, and move to the next level with the same eps.
    Returns the final samples and the list of (t, I_t) at every level."""
    if n_steps < 1:
        raise ValueError("n_steps must be at least 1")
    T = len(gammas)
    levels = [int(round(T * (1.0 - k / n_steps))) for k in range(n_steps + 1)]
    I = np.array(I_T, dtype=float)
    history = [(levels[0], I.copy())]
    for t, t_next in zip(levels[:-1], levels[1:]):
        g, g_next = gamma_at(t, gammas), gamma_at(t_next, gammas)
        I0_hat = oracle_I0(I, g)
        eps_hat = (I - np.sqrt(g) * I0_hat) / np.sqrt(max(1.0 - g, 1e-12))
        I = np.sqrt(g_next) * I0_hat + np.sqrt(1.0 - g_next) * eps_hat
        history.append((t_next, I.copy()))
    return I, history


def plot_histograms(snapshots: dict, gammas, bins=np.linspace(-4, 4, 81)):
    """Histograms of I_t for the levels t of the dict t -> samples, one panel each (same bins; own density axis)."""
    fig, axes = plt.subplots(1, len(snapshots), figsize=(2.8 * len(snapshots), 2.6))
    for ax, (t, samples) in zip(np.atleast_1d(axes), snapshots.items()):
        ax.hist(samples, bins=bins, density=True, color=COLOUR["histogram"])
        ax.set_title(rf"$t = {t}$, $\gamma_t = {gamma_at(t, gammas):.3g}$", fontsize=9)
        ax.set_xlabel(r"$I_t$")
    np.atleast_1d(axes)[0].set_ylabel("density")
    fig.tight_layout()
    plt.show()


def plot_schedule(gammas):
    """gamma_t, sqrt(gamma_t) (the weight of I_0) and sqrt(1 - gamma_t) (the weight of the noise) against t."""
    t = np.arange(1, len(gammas) + 1)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(t, gammas, label=r"$\gamma_t$")
    ax.plot(t, np.sqrt(gammas), label=r"$\sqrt{\gamma_t}$")
    ax.plot(t, np.sqrt(1 - gammas), label=r"$\sqrt{1 - \gamma_t}$")
    ax.set_xlabel("step $t$")
    ax.set_ylabel("value")
    ax.set_title("the linear schedule")
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


def plot_reconstruction(results: dict, bins=np.linspace(-1.5, 1.5, 121)):
    """Histograms of reconstructed samples, one panel per entry of the dict title -> samples."""
    fig, axes = plt.subplots(1, len(results), figsize=(4.2 * len(results), 2.8), sharey=True)
    for ax, (title, samples) in zip(np.atleast_1d(axes), results.items()):
        ax.hist(samples, bins=bins, density=True, color=COLOUR["histogram"])
        ax.set_title(title, fontsize=10)
        ax.set_xlabel(r"reconstructed $I_0$")
    np.atleast_1d(axes)[0].set_ylabel("density")
    fig.tight_layout()
    plt.show()


# ------------------------------------------------------------------ checks of the utilities

_x_check = np.array([0.3, -0.7, 1.1])
assert np.allclose(jacobian(_x_check), finite_difference_jacobian(_x_check), atol=1e-6), "analytic Jacobian disagrees with finite differences"
_g_check, _I_check = 0.5241, np.linspace(-3, 3, 13)
_p_plus = np.exp(-(_I_check - np.sqrt(_g_check)) ** 2 / (2 * (1 - _g_check)))
_p_minus = np.exp(-(_I_check + np.sqrt(_g_check)) ** 2 / (2 * (1 - _g_check)))
assert np.allclose((_p_plus - _p_minus) / (_p_plus + _p_minus), oracle_I0(_I_check, _g_check), atol=1e-12), "oracle_I0 disagrees with Bayes' rule"
print("utilities loaded; the analytic Jacobian and oracle_I0 pass their checks")

## Three optimisers on the toy arm

Fitting a model to $N$ observed image features $\mathbf r_i$ minimises the squared distances to the features $\tilde{\mathbf r}_i(\mathbf x)$ it predicts (the notes for Lecture 18),

$$e(\mathbf x) = \sum_{i=1}^{N} \|\tilde{\mathbf r}_i(\mathbf x) - \mathbf r_i\|^2 = \mathbf e^T \mathbf e, \qquad \mathbf e \in \mathbb R^{2N} \text{ the stacked residuals};$$

here the parameters are $\mathbf x$ (one arm, no image index $t$). With $\mathbf J$ the $2N \times d$ Jacobian of $\mathbf e$, $\mathbf e(\mathbf x + \Delta\mathbf x) \approx \mathbf e + \mathbf J \Delta\mathbf x$, and the three methods of the lecture step by

- Gauss-Newton: $\mathbf J^T \mathbf J \, \Delta\mathbf x = -\mathbf J^T \mathbf e$;
- Levenberg-Marquardt: $\Delta\mathbf x = -(\mathbf J^T \mathbf J + \mu \mathbf I)^{-1} \mathbf J^T \mathbf e$, $\mu$ lowered after a step that reduces $e$ and raised, the step tried again, after one that does not ($\mu_0 = 1$, factors of 10);
- gradient descent: $\Delta\mathbf x = -\eta \, \mathbf J^T \mathbf e$ with the fixed step $\eta = 0.15$.

Least squares is the maximum likelihood estimate for independent Gaussian errors; a cost that grows more slowly than $r^2$ for large errors corresponds to an error distribution with heavier tails, such as the Geman-McClure penalty $\rho(r) = \sigma^2 r^2 / (\sigma^2 + r^2)$ per coordinate, which `robust_fit` minimises over the six residual coordinates by gradient descent ($\rho'(r) = 2\sigma^4 r / (\sigma^2 + r^2)^2$).

**The toy arm.** Three links of lengths 1, 0.8 and 0.6 from a fixed shoulder at the origin; $\mathbf x$ holds the three joint angles, each relative to the previous link; the features are the ends of the three links ($N = d = 3$). The observations `obs` are the link ends at $\mathbf x^* = (0.5, 0.9, -0.8)$ rad plus Gaussian noise of s.d. 0.03 on each of the six coordinates; every method starts from $\mathbf x_0 = (-2, 1.5, 2)$ and runs 12 iterations. The cell prints $e$ per iteration and the $\mu$ of each accepted Levenberg-Marquardt step, and draws the arm during that fit.

In [ ]:
rng = np.random.default_rng(0)
obs = arm_points(X_TRUE) + rng.normal(0.0, 0.03, size=(3, 2))        # observed link ends: truth plus noise, seed 0
noise_sq = float(np.sum((obs - arm_points(X_TRUE)) ** 2))             # the squared length of the noise added

e_gn, path_gn = gauss_newton(obs, X_START, n_iter=12)
e_lm, path_lm, mus, retries = levenberg_marquardt(obs, X_START, n_iter=12, mu0=1.0, factor=10.0)
e_gd, path_gd = gradient_descent(obs, X_START, n_iter=12, step=0.15)

print(f"e at iterations 0..12 (start e(x_0) = {e_gn[0]:.4g})")
print(f"  Gauss-Newton:        {fmt(e_gn)}")
print(f"  Levenberg-Marquardt: {fmt(e_lm)}")
print(f"  gradient descent:    {fmt(e_gd)}")
print(f"mu of each accepted Levenberg-Marquardt step: {fmt(mus, 3)}   (retries before it: {' '.join(str(r) for r in retries)})")
print(f"final e: Gauss-Newton {e_gn[-1]:.4g}, Levenberg-Marquardt {e_lm[-1]:.4g}, gradient descent {e_gd[-1]:.4g};   noise_sq = {noise_sq:.4g}")
print(f"Levenberg-Marquardt's final angles {fmt(path_lm[-1])} (true {fmt(X_TRUE)})")

plot_errors({"Gauss-Newton": e_gn, "Levenberg-Marquardt": e_lm, "gradient descent": e_gd})
plot_arm_fit(path_lm, obs)

### Question 11.1 (4 points)

- Show the plot of $e$ against the iteration for the three methods and report Gauss-Newton's first two values of $e$. **(1 point)**
- Why can the first Gauss-Newton step increase $e$, while Levenberg-Marquardt's rule for $\mu$ guarantees that $e$ never increases? **(1 point)**
- Report the final $e$ of the three methods next to `noise_sq`, the squared length of the noise added to the observations. Why do Gauss-Newton and Levenberg-Marquardt stop at the same value, why is it not zero, and why is it smaller than `noise_sq`? **(1 point)**
- Move the observed end of the third link by 1.5 units in $x$ (`obs_outlier` below); fit with the squared cost (`levenberg_marquardt`) and, starting from that fit, with the Geman-McClure cost (`robust_fit`, $\sigma = 0.3$). Show the two fitted arms over the true one (`plot_arms`) and report the position errors of the elbow and the wrist, the ends of the first two links (`joint_errors`), under each cost. **(1 point)**

In [ ]:
# Your code for Question 11.1, e.g.
# obs_outlier = obs.copy()
# obs_outlier[2, 0] += 1.5                                        # the observed end of the third link, moved by 1.5 in x
# e_sq, path_sq, _, _ = levenberg_marquardt(obs_outlier, X_START)
# x_sq = path_sq[-1]
# x_rob, costs_rob = robust_fit(obs_outlier, x_sq, sigma=0.3)
# plot_arms({"squared cost": x_sq, "Geman-McClure": x_rob}, obs_outlier)
# print("squared cost: elbow, wrist, end of link 3 errors", fmt(joint_errors(x_sq)))
# print("Geman-McClure:", fmt(joint_errors(x_rob)))

**Your answers for Question 11.1:**

## Toy render-and-compare with an occluder

A neural mesh model $\mathfrak N_y$ is a mesh whose vertex $r$ carries a feature vector $\theta_r$ instead of a colour (the notes for Lecture 19). Rendered at the camera pose $m$ it predicts a feature map, compared with the feature map $F$ of the image: given mesh and pose the features $f_i$ are independent, a foreground position $i$ (covered by the mesh) being explained by the vertex $r$ rendered there and a background position $i'$ by the one clutter feature $\beta$ of the background model $B$,

$$p(F \mid \mathfrak N_y, m, B) = \prod_{i \in \mathcal{FG}} p(f_i \mid \mathfrak N_y, m) \prod_{i' \in \mathcal{BG}} p(f_{i'} \mid B),$$

both factors Gaussian; with all spreads 1 and the constants dropped, the negative log-likelihood is

$$\mathcal L_{ML}(m) = \tfrac12 \sum_{i \in \mathcal{FG}} \|f_i - \theta_r\|^2 + \tfrac12 \sum_{i' \in \mathcal{BG}} \|f_{i'} - \beta\|^2 .$$

Render-and-compare renders $\mathfrak N_y$ at $m$, compares with $F$ and changes $m$ to reduce the loss (a gradient step if the renderer is differentiable, a search otherwise), starting from the best of poses sampled uniformly. For occluders, each foreground position gets a binary variable $z_i$ with prior $p(z_i = 1) = \pi$: the foreground score $\log p(f_i \mid \mathfrak N_y, m) + \log \pi$ competes with the background score $\log p(f_i \mid B) + \log(1 - \pi)$, the larger wins, and the map $Z$ of the $z_i$ is the occlusion prediction (the notes' colours: yellow background, green visible object, red predicted occluded). Pose accuracy is measured three ways: $\mathrm{ACC}_{\pi/6}$ and $\mathrm{ACC}_{\pi/18}$, the percentages of images whose pose error is below $\pi/6$ and $\pi/18$, and MedErr, the median pose error in degrees.

**The toy.** $K = 12$ vertices on an irregular polygon around the centre of a $24 \times 24$ grid, each with a fixed random feature $\theta_r \in \mathbb R^8$ of length 2; $\beta = \mathbf 0$; the pose $m$ is one rotation angle $\varphi \in [-180°, 180°)$ (position and scale given). `render` gives every cell inside the polygon the feature of its nearest vertex and every other cell $\beta$; a target $F$ is the rendering at $\varphi^*$ plus Gaussian noise of s.d. 0.3 on every cell. `search_pose` moves $\varphi$ by steps of 4°, halved whenever neither neighbour improves the loss, until the step is below 0.25°; `best_of_starts` evaluates the loss at 12 angles 30° apart and searches from the best. The second cell pastes a square occluder of random unit features over part of the object and applies the $z_i$ rule (`nll_with_z`). With another occluder (seed or level) the two estimates may coincide: the pose is a single angle, which the visible cells alone can fix when the occluder's features cost about the same at every pose; the $z_i$ rule's gain then shows in the map $Z$ and in the loss at the true pose rather than in the pose.

In [ ]:
mesh = make_mesh(K=12, D=8, seed=0)
phi_true = 40.0
F_clean, inside_true = make_target(mesh, phi_true, noise=0.3, seed=1)
n_object = int(inside_true.sum())
print(f"the object covers {n_object} of the {GRID * GRID} cells at phi* = {phi_true:g} deg")
show_feature_norm(F_clean, f"the clean target: |f_i| (phi* = {phi_true:g} deg)", mesh=mesh, phi=phi_true)

angles = np.arange(-180.0, 180.0, 1.0)
losses = np.array([nll(F_clean, mesh, a) for a in angles])
print(f"loss on the 1-degree grid: minimum {losses.min():.2f} at phi = {angles[np.argmin(losses)]:g} deg "
      f"(loss at the true angle {nll(F_clean, mesh, phi_true):.2f}; largest value {losses.max():.2f})")
plot_loss_curve(angles, losses, phi_true)

starts = wrap(np.arange(12) * 30.0)
results = [search_pose(F_clean, mesh, float(s)) for s in starts]
errors_starts = np.array([pose_error(phi, phi_true) for phi, _, _ in results])
print("search from 12 starts (start -> estimate, pose error):")
print("  " + "; ".join(f"{s:g} -> {phi:.2f} ({err:.1f})" for s, (phi, _, _), err in zip(starts, results, errors_starts)))
print(f"  {int(np.sum(errors_starts < 5.0))} of the 12 starts end within 5 degrees of the truth")
phi_best, loss_best, n_eval = best_of_starts(F_clean, mesh, n_starts=12)
print(f"best of 12 starts: phi = {phi_best:.2f} deg, pose error {pose_error(phi_best, phi_true):.2f} deg, {n_eval} loss evaluations after the 12")

rng = np.random.default_rng(100)
errors_single, errors_best = [], []
for k in range(50):                                       # 50 random targets, each with its own angle and noise
    phi_k = float(rng.uniform(-180.0, 180.0))
    F_k, _ = make_target(mesh, phi_k, noise=0.3, seed=1000 + k)
    errors_single.append(pose_error(search_pose(F_k, mesh, 0.0)[0], phi_k))
    errors_best.append(pose_error(best_of_starts(F_k, mesh, n_starts=12)[0], phi_k))
print("over 50 random targets:")
for name, errs in (("one start at 0 deg", errors_single), ("best of 12 starts", errors_best)):
    m = accuracy_measures(errs)
    print(f"  {name:20s} ACC_pi/6 = {m['ACC_pi/6']:.0f} %, ACC_pi/18 = {m['ACC_pi/18']:.0f} %, MedErr = {m['MedErr']:.2f} deg")

In [ ]:
OCCLUSION_LEVEL = 0.6           # fraction of the object's cells to cover (the square grows in rings, so it overshoots a little)
PI = 0.6                        # the prior p(z_i = 1)
F_occ, occluded, square, level_used = add_occluder(F_clean, inside_true, level=OCCLUSION_LEVEL, noise=0.3, seed=17)
print(f"occluder: a square of {int(square.sum())} cells with random unit features; it covers {int(occluded.sum())} of the object's "
      f"{n_object} cells, an occlusion level of {100 * level_used:.1f} %")
show_feature_norm(F_occ, "the occluded target: |f_i|", mesh=mesh, phi=phi_true)

loss_z = lambda F, mesh, phi: nll_with_z(F, mesh, phi, pi=PI)[0]
phi_ml, _, _ = best_of_starts(F_occ, mesh, n_starts=12, loss=nll)
phi_z, _, _ = best_of_starts(F_occ, mesh, n_starts=12, loss=loss_z)
print(f"plain L_ML:           phi = {phi_ml:.2f} deg, pose error {pose_error(phi_ml, phi_true):.2f} deg")
print(f"z_i rule (pi = {PI}): phi = {phi_z:.2f} deg, pose error {pose_error(phi_z, phi_true):.2f} deg")

_, Z = nll_with_z(F_occ, mesh, phi_z, pi=PI)
Z_true = np.full((GRID, GRID), 2)
Z_true[inside_true] = 1
Z_true[occluded] = 0
red_fraction = float(np.mean(Z[occluded] == 0))
print(f"Z marks {100 * red_fraction:.1f} % of the {int(occluded.sum())} occluded cells red "
      f"and {100 * np.mean(Z[inside_true & ~occluded] == 0):.1f} % of the visible object cells")
plot_z({f"Z at the estimated pose ({phi_z:.1f} deg)": Z, "true occluder mask": Z_true})

### Question 11.2 (4 points)

- Show the loss curve $\mathcal L_{ML}(\varphi)$ of the clean target with the true angle marked, and report the angle of its minimum. **(1 point)**
- Report how many of the 12 starts end within 5° of the truth, and the three accuracy measures over the 50 random targets for the single start at 0° and for the best of 12 starts. Why does the search start from the best of several poses? **(1 point)**
- With the occluder, report the pose error of plain $\mathcal L_{ML}$ and of the $z_i$ rule ($\pi = 0.6$), and the occlusion level the cell used. **(1 point)**
- Show the map $Z$ next to the true occluder mask in the colour code of the notes, and report the fraction of the occluded cells that $Z$ marks red. **(1 point)**

In [ ]:
# Your code for Question 11.2 (the two cells above print every number asked; use this cell if you vary pi, the level or the seeds), e.g.
# F_occ2, occluded2, square2, level2 = add_occluder(F_clean, inside_true, level=0.4, seed=17)

**Your answers for Question 11.2:**

## Diffusing an image into noise, and back

In the formulation of Ho et al. (the notes for Lecture 15), each of 1000 steps shrinks the image slightly and adds a little Gaussian noise, so that after $t$ steps

$$I_t = \sqrt{\gamma_t}\, I_0 + \sqrt{1 - \gamma_t}\, \epsilon, \qquad \epsilon \sim \mathcal N(0, \mathrm{Id}), \qquad \gamma_t = \prod_{s=1}^{t} (1 - \beta_s),$$

with small noise variances $\beta_s$, here rising linearly from $10^{-4}$ to $0.02$; a network $\epsilon_\omega$ learns to predict the noise from $I_t$ and $t$, and generation starts from noise and removes the predicted noise step by step.

**The toy.** The "image" is a single number $I_0 = \pm 1$ with probability $\tfrac12$ each (20 000 samples), so the expected value of $I_0$ given $I_t$ is known exactly: Bayes' rule on the two values gives $\hat I_0 = \mathbb E[I_0 \mid I_t] = (p_+ - p_-)/(p_+ + p_-)$ with $p_\pm \propto e^{-(I_t \mp \sqrt{\gamma_t})^2 / (2 (1 - \gamma_t))}$, that is $\hat I_0 = \tanh\big(\sqrt{\gamma_t}\, I_t / (1 - \gamma_t)\big)$ (`oracle_I0`). Solving the formula above for the noise, $\hat\epsilon = (I_t - \sqrt{\gamma_t}\, \hat I_0) / \sqrt{1 - \gamma_t}$, and a sample moves to a less noisy level $t' < t$ by $I_{t'} = \sqrt{\gamma_{t'}}\, \hat I_0 + \sqrt{1 - \gamma_{t'}}\, \hat\epsilon$; `reconstruct` repeats this from $t = 1000$ (pure noise) down to $t = 0$ in `n_steps` equal steps of $t$.

In [ ]:
T = 1000
betas, gammas = linear_schedule(T=T, beta_1=1e-4, beta_T=0.02)
print("gamma_t:", ", ".join(f"t = {t}: {gamma_at(t, gammas):.4g}" for t in (100, 250, 500, 1000)))
t_half = int(np.argmax(gammas < 0.5)) + 1
print(f"first t with gamma_t < 1/2: {t_half}")

rng = np.random.default_rng(0)
n_samples = 20_000
I0 = rng.choice([-1.0, 1.0], size=n_samples)                        # the two-valued "image", unit variance
snapshots = {t: diffuse(I0, t, gammas, rng) for t in (0, 100, 250, 500, 1000)}
plot_histograms(snapshots, gammas)
plot_schedule(gammas)

deviations = np.array([abs(diffuse(I0, t, gammas, rng).var() - 1.0) for t in range(1, T + 1)])
print(f"largest |Var(I_t) - 1| over t = 1..{T}: {deviations.max():.4f} (at t = {int(np.argmax(deviations)) + 1}); "
      f"at t = 100, 250, 500, 1000: {fmt([abs(snapshots[t].var() - 1.0) for t in (100, 250, 500, 1000)], 2)}")

In [ ]:
I_T = rng.normal(size=n_samples)                                     # pure noise at t = 1000
samples_30, history_30 = reconstruct(I_T, gammas, n_steps=30)
within_30 = float(np.mean(np.abs(np.abs(samples_30) - 1.0) < 0.05))
print(f"30 steps (t = {', '.join(str(t) for t, _ in history_30[:4])}, ..., {history_30[-1][0]}): "
      f"{100 * within_30:.2f} % of the samples within 0.05 of +-1; mean {samples_30.mean():+.4f}, "
      f"fraction positive {np.mean(samples_30 > 0):.4f}")
plot_reconstruction({"reconstructed in 30 steps": samples_30})

### Question 11.3 (2 points)

- Report $\gamma_t$ at $t = 100, 250, 500, 1000$ and the first $t$ with $\gamma_t < 1/2$, show the five histograms of $I_t$, and report the largest deviation of $\mathrm{Var}(I_t)$ from 1 over $t = 1, \ldots, 1000$. **(1 point)**
- Show the histograms of the reconstructed samples after 30 steps and after 10 steps (`reconstruct(I_T, gammas, n_steps=10)`), report the fraction within 0.05 of $\pm 1$ in each case, and say in one sentence what this exact $\hat I_0$ replaces in the lecture's description of diffusion models. **(1 point)**

In [ ]:
# Your code for Question 11.3, e.g.
# samples_10, history_10 = reconstruct(I_T, gammas, n_steps=10)
# print(f"10 steps: {100 * np.mean(np.abs(np.abs(samples_10) - 1.0) < 0.05):.2f} % within 0.05 of +-1")
# plot_reconstruction({"reconstructed in 30 steps": samples_30, "reconstructed in 10 steps": samples_10})

**Your answers for Question 11.3:**